# 1. 라이브러리 및 환경 설정

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
import torchaudio
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

CFG = {
    'IMG_SIZE' : 224,
    'EPOCHS' : 20,
    'LEARNING_RATE' : 1e-4,
    'BATCH_SIZE' : 32,
    'SEED' : 42,
    'WEIGHT_DECAY' : 1e-2
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. 데이터 로드 및 학습/검증/테스트 데이터 분할

In [2]:
train_df = pd.read_csv('./train.csv')
val_df = pd.read_csv('./dev.csv')
test_df = pd.read_csv('./sample_submission.csv')

print(f"학습 데이터 차원 : {train_df.shape}")
print(f"검증 데이터 차원 : {val_df.shape}")
print(f"테스트 데이터 차원 : {test_df.shape}")

학습 데이터 차원 : (1000, 2)
검증 데이터 차원 : (100, 2)
테스트 데이터 차원 : (1000, 3)


In [3]:
train_df['label'].value_counts()

label
unstable    500
stable      500
Name: count, dtype: int64

# 커스텀 데이터셋 클래스 정의 및 데이터 로더

In [4]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable' : 0 , 'unstable' : 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        #폴더 경로 설정
        sample_id = str(self.df.iloc[idx]['id'])
        folder_path = os.path.join(self.root_dir, sample_id)

        #전면, 윗면 뷰 로드
        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        #테스트 모드일 때 이미지 리스트만 반환
        if self.is_test:
            return views

        #학습/검증 모드일 때 라벨 함께 반환
        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

In [5]:
train_transform = transforms.Compose([
    #데이터 증강
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

#학습/검증 세트 준비
train_dataset = MultiViewDataset(train_df, './train', train_transform, is_test = False)
val_dataset = MultiViewDataset(val_df, './dev', test_transform, is_test = False)

train_loader = DataLoader(train_dataset, batch_size = CFG['BATCH_SIZE'], shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = CFG['BATCH_SIZE'], shuffle = False)

test_dataset = MultiViewDataset(test_df, './test', test_transform, is_test = True)
test_loader = DataLoader(test_dataset, batch_size = CFG['BATCH_SIZE'], shuffle = False)

# 4. 모델 정의

In [6]:
class MultiViewResNet(nn.Module):
    def __init__(self, num_classes=1):
        super(MultiViewResNet, self).__init__()
        self.backbone = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Linear(512 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, views):
        #views: [batch, 3, 224, 224] * 2 images
        f1 = self.feature_extractor(views[0]).view(views[0].size(0), -1)
        f2 = self.feature_extractor(views[1]).view(views[1].size(0), -1)

        combined = torch.cat((f1, f2), dim = 1)
        return self.classifier(combined)

# 5. 학습 및 검증 루프 구현

In [7]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    for views, labels in tqdm(loader, desc = "Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc = "Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views).view(-1)
            #sigmoid 함수로 unstable할 확률값으로 변환
            probs = torch.sigmoid(outputs)
            probs = torch.clamp(probs, 1e-7, 1 - 1e-7)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype = np.float64)
    all_labels = np.array(all_labels, dtype = np.float64)

    #대회 공식 Log Loss 계산
    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    #Binary Log Loss
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))

    #Accuracy 계산
    acc_score = np.mean((all_probs > 0.5) == all_labels)

    return logloss_score, acc_score

In [8]:
model = MultiViewResNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr = CFG['LEARNING_RATE'], weight_decay = CFG['WEIGHT_DECAY'])
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=1, eta_min=1e-6)
best_val_loss = float('inf')

#-- Main Loop --
for epoch in range(1, CFG['EPOCHS'] + 1):
    avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_logloss, val_acc = validate(model, val_loader, criterion, device)
    #scheduler 업데이트
    scheduler.step()

    #현재 epoch 학습률 확인
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch[{epoch}] - LR: {current_lr:.6f}")
    print(f" -Train Loss: {avg_train_loss:.4f}")
    print(f" -Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}")

    if val_logloss < best_val_loss:
        best_val_loss = val_logloss
        torch.save(model.state_dict(), 'best_model.pth')
        print(f" >>> Best Model Saved! (Loss: {best_val_loss:.6f})")

model.load_state_dict(torch.load('best_model.pth'))
print("최종 예측을 위한 best model을 로드했습니다.")

Training:   0%|          | 0/32 [00:00<?, ?it/s]

Validation:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch[1] - LR: 0.000091
 -Train Loss: 0.1974
 -Val Log-Loss: 0.528748 | Val Acc: 0.7400
 >>> Best Model Saved! (Loss: 0.528748)


Training:   0%|          | 0/32 [00:00<?, ?it/s]

Validation:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch[2] - LR: 0.000066
 -Train Loss: 0.0020
 -Val Log-Loss: 0.813628 | Val Acc: 0.6300


... (50 outputs left out)

Epoch[19] - LR: 0.000010
 -Train Loss: 0.0001
 -Val Log-Loss: 0.780115 | Val Acc: 0.7300


Training:   0%|          | 0/32 [00:00<?, ?it/s]

Validation:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch[20] - LR: 0.000100
 -Train Loss: 0.0001
 -Val Log-Loss: 0.598753 | Val Acc: 0.7600
최종 예측을 위한 best model을 로드했습니다.


# 6. 추론 및 제출 파일 생성

In [9]:
model.eval()
all_probs = []

with torch.no_grad():
    for views in tqdm(test_loader, desc = "Inference"):
        views = [v.to(device) for v in views]

        #모델 출력(Logit -> sigmoid -> unstable할 확률)
        outputs = model(views).view(-1)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())
        
all_probs = np.array(all_probs)
submission = pd.DataFrame({
    'id' : test_df['id'],
    'unstable_prob' : all_probs,
    'stable_prob' : 1.0 - all_probs
})

submission.to_csv('submission.csv', encoding = 'utf-8-sig', index = False)
print("submission.csv 저장 완료")

Inference:   0%|          | 0/32 [00:00<?, ?it/s]

submission.csv 저장 완료
